# Comparison of Three RAG Methods on 30 HotpotQA Samples

This notebook compares:
1. Simple RAG
2. Multi-hop RAG with Query Decomposition
3. Multi-hop RAG with Sequential Decomposition and Iterative Retrieval

The notebook aligns records by normalized question text, calculates Exact Match, Flexible Match, and Token F1, highlights cases for semantic review, and summarizes available runtime and call-count information.

**Before running:** Mount Google Drive, then update the input paths in the configuration cell. The output directory can also be changed there.

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import json, re, unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

DRIVE_ROOT = Path("/content/drive/MyDrive")
BASIC_RAG_ROOT = DRIVE_ROOT / "Basic_RAG"
MULTIHOP_RAG_ROOT = DRIVE_ROOT / "Multi_hop_RAG"

SIMPLE_FILE = BASIC_RAG_ROOT / "results/hotpotqa" / "simple_rag_results_30samples.json"
DECOMPOSITION_FILE = MULTIHOP_RAG_ROOT / "results/hotpotqa" / "rag_evaluation_results_30samples.csv"
ITERATIVE_FILE = MULTIHOP_RAG_ROOT / "results/hotpotqa" / "debugging_results_30_iterative_retrieval.json"
OUTPUT_DIR = MULTIHOP_RAG_ROOT / "results/hotpotqa"

# Validate paths before loading the input files.
for file_path in [SIMPLE_FILE, DECOMPOSITION_FILE, ITERATIVE_FILE]:
    print(f"{file_path}: {'FOUND' if file_path.exists() else 'NOT FOUND'}")

missing = [str(p) for p in [SIMPLE_FILE, DECOMPOSITION_FILE, ITERATIVE_FILE] if not p.exists()]
if missing:
    raise FileNotFoundError("Update the input paths; files not found: " + ", ".join(missing))

with open(SIMPLE_FILE, "r", encoding="utf-8") as f:
    simple = json.load(f)
decomp = pd.read_csv(DECOMPOSITION_FILE)
with open(ITERATIVE_FILE, "r", encoding="utf-8") as f:
    iterative = json.load(f)

print("Simple RAG records:", len(simple))
print("Query Decomposition records:", len(decomp))
print("Iterative Retrieval records:", len(iterative))
print("CSV columns:", decomp.columns.tolist())

/content/drive/MyDrive/Basic_RAG/results/hotpotqa/simple_rag_results_30samples.json: FOUND
/content/drive/MyDrive/Multi_hop_RAG/results/hotpotqa/rag_evaluation_results_30samples.csv: FOUND
/content/drive/MyDrive/Multi_hop_RAG/results/hotpotqa/debugging_results_30_iterative_retrieval.json: FOUND
Simple RAG records: 30
Query Decomposition records: 30
Iterative Retrieval records: 30
CSV columns: ['Question', 'Gold Answer', 'Generated Answer', 'Retrieved Docs', 'Exact Match', 'Generation Time (s)', 'Total Time (s)', 'LLM Calls', 'Retrieval Calls']


## 1. Text Normalization and Evaluation Metrics

- **Exact Match (EM):** Checks whether the normalized prediction exactly matches the normalized reference answer.
- **Flexible Match:** Checks whether either normalized answer phrase appears inside the other. This is a lexical screening metric and can produce false positives.
- **Token F1:** Measures token overlap between the prediction and reference; it does not prove semantic correctness.

For ambiguous cases, inspect the question, reference answer, and generated answer together or use a consistent semantic-judging rubric.

In [4]:
def norm(x):
    if x is None or (isinstance(x,float) and np.isnan(x)):
      return ''
    x=unicodedata.normalize('NFKC',str(x)).lower().strip()
    return re.sub(r'\s+',' ',re.sub(r'[^\w\s]',' ',x)).strip()

def em(pred,gold):
  return int(bool(norm(gold)) and norm(pred)==norm(gold))


def flexible_match(pred,gold):
    p,g=norm(pred),norm(gold)
    if not p or not g:
      return 0
    return int(p==g or re.search(r'(?<!\w)'+re.escape(p)+r'(?!\w)',g) is not None or re.search(r'(?<!\w)'+re.escape(g)+r'(?!\w)',p) is not None)

def token_f1(pred,gold):
    from collections import Counter
    p,g=norm(pred).split(),norm(gold).split()

    if not p or not g:
      return 0.0

    overlap=sum((Counter(p)&Counter(g)).values())
    if not overlap:
      return 0.0

    pr,rc=overlap/len(p),overlap/len(g)
    return 2*pr*rc/(pr+rc)

def pick(row,keys,default=''):
    for k in keys:
        if k in row and row[k] is not None and not (isinstance(row[k],float) and np.isnan(row[k])):
          return row[k]
    return default


def qkey(q):
  return norm(q)


def fnum(x):
    try:
      return float(x)
    except (TypeError,ValueError):
      return np.nan

## 2. Align Records Across the Three Methods

Records are aligned using normalized question text rather than row order. If the printed CSV column names differ from the candidate names in the code, update the relevant field-name lists.

In [6]:
sm={qkey(pick(r,['question','query'])):r for r in simple if pick(r,['question','query'])}
dm={qkey(pick(r,['Question','question','Query','query'])):r.to_dict() for _,r in decomp.iterrows() if pick(r,['Question','question','Query','query'])}
im={qkey(pick(r,['question','query'])):r for r in iterative if pick(r,['question','query'])}

keys=list(dict.fromkeys(list(sm)+list(dm)+list(im)))
rows=[]

for k in keys:
    s,d,it=sm.get(k,{}),dm.get(k,{}),im.get(k,{})

    q=pick(s,['question'],pick(d,['Question','question'],pick(it,['question'],k)))
    gold=pick(s,['reference_answer','gold_answer','answer'],pick(d,['Gold Answer','Reference Answer','reference_answer','gold_answer']))

    rows.append({'Question':q,'Gold Answer':gold,
      'Simple RAG Answer':pick(s,['generated_answer','final_answer']),
      'Query Decomposition Answer':pick(d,['Generated Answer','generated_answer','Final Answer','final_answer']),
      'Iterative Retrieval Answer':pick(it,['final_answer','generated_answer','answer']),
      'Simple RAG Time (s)':fnum(pick(s,['generation_time_seconds','elapsed_seconds'],np.nan)),
      'Query Decomposition Total Time (s)':fnum(pick(d,['Total Time (s)','Total Time','total_time_seconds'],np.nan)),
      'Query Decomposition LLM Calls':fnum(pick(d,['LLM Calls','llm_calls'],np.nan)),
      'Query Decomposition Retrieval Calls':fnum(pick(d,['Retrieval Calls','retrieval_calls'],np.nan)),
      'Iterative Hops':fnum(pick(it,['num_hops'],np.nan)),
      'Iterative Retrieval Calls':fnum(pick(it,['num_retrievals'],np.nan)),
      'Iterative LLM Calls':fnum(pick(it,['num_llm_calls'],np.nan)),
      'Iterative Time (s)':fnum(pick(it,['elapsed_seconds'],np.nan)),
      'Simple RAG present':bool(s),'Query Decomposition present':bool(d),'Iterative Retrieval present':bool(it)})

comparison=pd.DataFrame(rows)

pres=['Simple RAG present','Query Decomposition present','Iterative Retrieval present']

print('Unique questions:',len(comparison),' | Present in all methods:',int(comparison[pres].all(axis=1).sum()))

if not comparison[pres].all(axis=1).all():
  display(comparison.loc[~comparison[pres].all(axis=1),['Question']+pres])

display(comparison[['Question','Gold Answer','Simple RAG Answer','Query Decomposition Answer','Iterative Retrieval Answer']])

Unique questions: 30  | Present in all methods: 30


,Question,Gold Answer,Simple RAG Answer,Query Decomposition Answer,Iterative Retrieval Answer
0,What was Iqbal F. Qadir on when he participate...,flotilla,Insufficient information.,Vice-Admiral Iqbal Fazl Quadir was part of the...,Cannot be determined from the evidence.
1,When did the park at which Tivolis Koncertsal ...,15 August 1843,15 August 1843,1843,"Tivoli Gardens, the park at which Tivolis Konc..."
2,What is the shared country of ancestry between...,Armenian,Insufficient information,Armenian,Armenia
3,The school in which the Wilmslow Show is held ...,Centre of Excellence,Centre of Excellence,good,The school in which the Wilmslow Show is held ...
4,Who will Billy Howle be seen opposite in the u...,Saoirse Ronan,Saoirse Ronan,Saoirse Ronan,Saoirse Ronan
5,"What animated movie, starring Danny Devito, fe...",The Lorax,Insufficient information.,Fire and Ice (1983 film),Cannot be determined from the evidence.
6,Out of the actors who have played the role of ...,Scott Adkins,Jean-Claude Van Damme,Scott Adkins,Scott Adkins
7,Tommy's Honour was a drama film that included ...,War & Peace,Insufficient information,"Document 1 Content: Jack Lowden, who played Yo...",Tommy's Honour was a drama film that included ...
8,Which rock band chose its name by drawing it o...,Midnight Oil,Midnight Oil,Midnight Oil,Midnight Oil chose its name by drawing it out ...
9,"""Tunak"", is a bhangra/pop love song by an arti...",1967,1960,1969,1967


## 3. Calculate Metrics and Summarize Results

In [7]:
answer_cols={'Simple RAG':'Simple RAG Answer','Query Decomposition':'Query Decomposition Answer','Iterative Retrieval':'Iterative Retrieval Answer'}
for method,col in answer_cols.items():
    comparison[f'{method} EM']=comparison.apply(lambda r:em(r[col],r['Gold Answer']),axis=1)
    comparison[f'{method} Flexible Match']=comparison.apply(lambda r:flexible_match(r[col],r['Gold Answer']),axis=1)
    comparison[f'{method} Token F1']=comparison.apply(lambda r:token_f1(r[col],r['Gold Answer']),axis=1)

summary=pd.DataFrame([{'Method':m,'N':len(comparison),'Exact Match (%)':round(100*comparison[f'{m} EM'].mean(),2),'Flexible Match (%)':round(100*comparison[f'{m} Flexible Match'].mean(),2),'Mean Token F1':round(comparison[f'{m} Token F1'].mean(),3)} for m in answer_cols])
display(summary)

,Method,N,Exact Match (%),Flexible Match (%),Mean Token F1
0,Simple RAG,30,33.33,40.00,0.367
1,Query Decomposition,30,30.00,40.00,0.346
2,Iterative Retrieval,30,26.67,46.67,0.369


## 4. Identify Cases for Manual or Semantic Review

This table highlights cases where Exact Match and Flexible Match disagree, as well as questions where the methods produce different outputs. Lexical metrics alone should not be treated as definitive semantic judgments.

In [9]:
comparison['Metric Disagreement']=np.logical_or.reduce([comparison[f'{m} EM']!=comparison[f'{m} Flexible Match'] for m in answer_cols])

cols=['Question','Gold Answer']

for m,a in answer_cols.items():
  cols += [a,f'{m} EM',f'{m} Flexible Match',f'{m} Token F1']

review=comparison.loc[comparison['Metric Disagreement'],cols]

print('Cases where EM and flexible match disagree:',len(review))
display(review)

Cases where EM and flexible match disagree: 8


,Question,Gold Answer,Simple RAG Answer,Simple RAG EM,Simple RAG Flexible Match,Simple RAG Token F1,Query Decomposition Answer,Query Decomposition EM,Query Decomposition Flexible Match,Query Decomposition Token F1,Iterative Retrieval Answer,Iterative Retrieval EM,Iterative Retrieval Flexible Match,Iterative Retrieval Token F1
0,What was Iqbal F. Qadir on when he participate...,flotilla,Insufficient information.,0,0,0.0,Vice-Admiral Iqbal Fazl Quadir was part of the...,0,1,0.105263,Cannot be determined from the evidence.,0,0,0.000000
1,When did the park at which Tivolis Koncertsal ...,15 August 1843,15 August 1843,1,1,1.0,1843,0,1,0.500000,"Tivoli Gardens, the park at which Tivolis Konc...",0,1,0.333333
3,The school in which the Wilmslow Show is held ...,Centre of Excellence,Centre of Excellence,1,1,1.0,good,0,0,0.000000,The school in which the Wilmslow Show is held ...,0,1,0.315789
7,Tommy's Honour was a drama film that included ...,War & Peace,Insufficient information,0,0,0.0,"Document 1 Content: Jack Lowden, who played Yo...",0,0,0.000000,Tommy's Honour was a drama film that included ...,0,1,0.160000
8,Which rock band chose its name by drawing it o...,Midnight Oil,Midnight Oil,1,1,1.0,Midnight Oil,1,1,1.000000,Midnight Oil chose its name by drawing it out ...,0,1,0.285714
11,In which film did Emilio Estevez star in in th...,The Outsiders,The Outsiders,1,1,1.0,The Outsiders,1,1,1.000000,"In 1983, Emilio Estevez starred in ""The Outsid...",0,1,0.400000
19,"What language, traditionally written with the ...",The Tugurt language,Tugurt,0,1,0.5,Berber,0,0,0.000000,Tugurt language,0,1,0.800000
23,George Gershwin is an American Composer and Ju...,a British composer,British,0,1,0.5,British,0,1,0.500000,Judith Weir is a composer from Britain.,0,0,0.400000


## 5. Compare Available Runtime and Call Counts

Missing measurements remain empty rather than being inferred. Runtime comparisons are meaningful only if timing boundaries and execution conditions were consistent across implementations.

In [10]:
cost = pd.DataFrame([
    {"Method": "Simple RAG", "Mean time (s)": comparison["Simple RAG Time (s)"].mean(),
     "Mean LLM calls": np.nan, "Mean retrieval calls": np.nan},
    {"Method": "Query Decomposition", "Mean time (s)": comparison["Query Decomposition Total Time (s)"].mean(),
     "Mean LLM calls": comparison["Query Decomposition LLM Calls"].mean(),
     "Mean retrieval calls": comparison["Query Decomposition Retrieval Calls"].mean()},
    {"Method": "Iterative Retrieval", "Mean time (s)": comparison["Iterative Time (s)"].mean(),
     "Mean LLM calls": comparison["Iterative LLM Calls"].mean(),
     "Mean retrieval calls": comparison["Iterative Retrieval Calls"].mean()},
])
display(cost.round(2))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
comparison_path = OUTPUT_DIR / "rag_three_methods_comparison.csv"
summary_path = OUTPUT_DIR / "rag_three_methods_metrics_summary.csv"
cost_path = OUTPUT_DIR / "rag_three_methods_cost_summary.csv"
comparison.to_csv(comparison_path, index=False, encoding="utf-8-sig")
summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
cost.to_csv(cost_path, index=False, encoding="utf-8-sig")
print("Saved output files:")
print(comparison_path)
print(summary_path)
print(cost_path)
print("Runtime comparisons are fair only if timing was measured consistently across all three methods.")

,Method,Mean time (s),Mean LLM calls,Mean retrieval calls
0,Simple RAG,1.40,NaN,NaN
1,Query Decomposition,9.44,2.00,1.90
2,Iterative Retrieval,22.88,3.73,1.87


Saved output files:
/content/drive/MyDrive/Multi_hop_RAG/results/hotpotqa/rag_three_methods_comparison.csv
/content/drive/MyDrive/Multi_hop_RAG/results/hotpotqa/rag_three_methods_metrics_summary.csv
/content/drive/MyDrive/Multi_hop_RAG/results/hotpotqa/rag_three_methods_cost_summary.csv
Runtime comparisons are fair only if timing was measured consistently across all three methods.
